[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_04_data_science_math/ch17_pandas_foundations/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 17 Hands-On Coding Exam — Clinical Tabular Data Pipelines (100 Points)

**Book Reference:** Chapter 17, Section 17.7 (Hands-On Coding Exam)  
**Suggested Time:** 40–50 minutes (Open Book, Closed Solution)

| Task | Pandas Data Engineering Pipeline | Points |
| :--- | :--- | :---: |
| **Task 17.1** | `prepare_triage_cohort(df)` (Non-mutating copy, `bmi`, `is_anemic`, adults `>= 18`, reset index) | 30 |
| **Task 17.2** | `rank_districts_by_urgent_share(df)` (District totals, urgent counts, 1-decimal `urgent_pct`, tie-broken sort) | 35 |
| **Task 17.3** | `reshape_and_compare_visits(wide_df)` (`pd.melt` to long format + 2-decimal mean paired hemoglobin change) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import pandas as pd
import sharukh_tutor as tutor

tutor.activate(chapter="ch17", notebook="coding_exam")


---
## Exam Task 17.1 (30 Points): Adult Triage Cohort Pipeline (`prepare_triage_cohort`)

Implement `prepare_triage_cohort(df)`:
1. Do **not** mutate the input DataFrame `df`.
2. Keep only adult patients (`age_years >= 18`).
3. Add `'bmi'` computed as `(weight_kg / (height_cm / 100) ** 2).round(2)`.
4. Add boolean column `'is_anemic'` (`hemoglobin_g_dl < 11.0`, strict `<`).
5. Reset the index with `drop=True` and return the prepared DataFrame.

In [ ]:
def prepare_triage_cohort(df):
    """Return non-mutating adult triage cohort with bmi and is_anemic."""
    adults = df.loc[df["age_years"] >= 18].copy()
    height_m = adults["height_cm"] / 100
    adults["bmi"] = (adults["weight_kg"] / (height_m ** 2)).round(2)
    adults["is_anemic"] = adults["hemoglobin_g_dl"] < 11.0
    return adults.reset_index(drop=True)


---
## Exam Task 17.2 (35 Points): District Urgent Share Ranking (`rank_districts_by_urgent_share`)

Implement `rank_districts_by_urgent_share(df)`:
1. Return a DataFrame with exact columns `["district", "total_patients", "urgent_patients", "urgent_pct"]`.
2. Preserve districts with **zero** Urgent patients (`urgent_patients == 0`, `urgent_pct == 0.0`).
3. Compute `urgent_pct` as `(urgent_patients / total_patients * 100).round(1)`.
4. Sort by `urgent_pct` **descending**, breaking ties by `district` **ascending**, and reset the index (`drop=True`) without mutating `df`.

In [ ]:
def rank_districts_by_urgent_share(df):
    """Rank districts by % Urgent patients desc, then district asc."""
    is_urgent = df["triage_priority"] == "Urgent"
    counts = pd.crosstab(df["district"], is_urgent)
    total = counts.sum(axis=1)
    urgent = (
        counts[True]
        if True in counts.columns
        else pd.Series(0, index=total.index)
    )
    out = pd.DataFrame(
        {
            "district": total.index,
            "total_patients": total.astype(int).values,
            "urgent_patients": urgent.astype(int).values,
        }
    )
    out["urgent_pct"] = (
        out["urgent_patients"] / out["total_patients"] * 100
    ).round(1)
    return out.sort_values(
        by=["urgent_pct", "district"],
        ascending=[False, True],
    ).reset_index(drop=True)


---
## Exam Task 17.3 (35 Points): Longitudinal Melt & Paired Change (`reshape_and_compare_visits`)

Implement `reshape_and_compare_visits(wide_df)`:
1. Given `wide_df` with columns `["patient_id", "hb_baseline", "hb_followup"]`, create a long melted DataFrame with exact columns `["patient_id", "visit", "hb_g_dl"]`.
2. Compute the mean paired hemoglobin change (follow-up minus baseline) as a native Python `float` rounded to **2 decimal places**: `float(round((wide_df["hb_followup"] - wide_df["hb_baseline"]).mean(), 2))`.
3. Return the 2-tuple `(long_df, mean_change)` without mutating `wide_df`.

In [ ]:
def reshape_and_compare_visits(wide_df):
    """Melt baseline/followup Hb and return (long_df, mean_change)."""
    long_df = pd.melt(
        wide_df,
        id_vars=["patient_id"],
        value_vars=["hb_baseline", "hb_followup"],
        var_name="visit",
        value_name="hb_g_dl",
    )
    deltas = wide_df["hb_followup"] - wide_df["hb_baseline"]
    mean_change = float(round(deltas.mean(), 2))
    return long_df, mean_change


In [ ]:
exam_submissions = {
    "ch17_exam_01": prepare_triage_cohort,
    "ch17_exam_02": rank_districts_by_urgent_share,
    "ch17_exam_03": reshape_and_compare_visits,
}

tutor.check_exam("ch17", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 17.1 (Strict Threshold & Non-Mutation):** Always copy before adding columns so caller DataFrames remain untouched, and check `hemoglobin_g_dl < 11.0` strictly (a reading of `11.0 g/dL` is `False`, not anemic).
- **Task 17.2 (Preserving Zero-Urgent Districts):** Filtering to `triage_priority == "Urgent"` first and then calling `.value_counts()` silently drops districts with `0` Urgent cases! Using `pd.crosstab(df["district"], df["triage_priority"] == "Urgent")` retains every district automatically.
- **Task 17.3 (Paired Change Direction):** Paired longitudinal improvement is always `hb_followup - hb_baseline` (positive when hemoglobin increases after iron supplementation; negative if it declined).